In [1]:
import os
import math
import json
import warnings
import itertools

warnings.filterwarnings("ignore")

In [2]:
os.environ.setdefault("FINTORCH_BASE_TIME_FRAME", "60")

'60'

In [3]:
import numpy as np 
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

from fintorch.enum import TimeFrame
from fintorch.exchange import ONLINE_EXCHANGE
from fintorch.utils.preprocess import remove_price_dependency
from fintorch.utils.args import DefaultArgumentParser
from fintorch.utils.plot import draw_candlestick_plot, draw_labels, draw_predictions

In [4]:
string_args = [
    "--start-date", "2024-01-01",
    "--stop-date", "2024-06-01",
    "--time-frame", "60",
]

args = DefaultArgumentParser.parse(args=string_args)

In [5]:
sdf = ONLINE_EXCHANGE.future.data.get_candles_dataframe(symbol=args.symbol, time_frame=args.time_frame).copy()
bdf = remove_price_dependency(df=sdf.copy())

bdf

,open,high,low,close,volume,trade
timestamp,,,,,,
1705395660,NaN,NaN,NaN,-0.000674,201.772,3323
1705395720,-0.000674,-0.000406,-0.001709,-0.001539,534.556,4093
1705395780,-0.001539,-0.001325,-0.001844,-0.001760,117.148,1847
1705395840,-0.001760,-0.001578,-0.002435,-0.002126,245.105,3502
1705395900,-0.002126,-0.001876,-0.002668,-0.002119,134.988,1928
...,...,...,...,...,...,...
1711395600,0.528236,0.528457,0.527553,0.527717,114.886,2182
1711395660,0.527717,0.529095,0.527488,0.528922,136.334,2193
1711395720,0.528922,0.530639,0.528922,0.529790,286.456,4391


In [87]:
LENGTH = 5
LOOK_AHEAD = 5

df = sdf[-10000:]

# add label
df["sma"] = df.close.rolling(window=LOOK_AHEAD, center=True).mean()
df["fsma"] = df.sma.shift(periods=-LOOK_AHEAD // 2)
df["label"] = df.sma < df.fsma

# method 1 **
features = ["mean-close", "std-close"]
df["mean-close"] = df.close.rolling(window=LENGTH).mean()
df["std-close"] = df.close.rolling(window=LENGTH).std()
    

# # method 2
# features = ["tr", "roc", "vpt"]
# df["tr"] = df.high - df.low
# df["roc"] = df.close - df.open
# df["vpt"] = df.volume / df.trade
    
df = df.dropna() 
df

,open,high,low,close,volume,trade,sma,fsma,label,mean-close,std-close
timestamp,,,,,,,,,,,
1710796140,67600.0,67600.1,67600.0,67600.0,52.173,515,67581.54,67574.64,False,67624.00,26.032768
1710796200,67600.0,67600.1,67441.1,67518.1,585.033,4136,67576.18,67600.30,True,67595.98,46.985072
1710796260,67518.0,67581.7,67507.7,67570.9,164.511,1785,67579.06,67616.50,True,67581.54,39.359027
1710796320,67571.0,67598.9,67536.4,67591.8,95.048,1768,67574.64,67626.04,True,67576.18,34.585792
1710796380,67591.8,67628.2,67577.3,67614.5,70.100,1342,67600.30,67633.00,True,67579.06,37.548276
...,...,...,...,...,...,...,...,...,...,...,...
1711395300,70665.6,70700.0,70665.5,70700.0,65.348,1648,70693.50,70676.50,False,70696.50,30.808765
1711395360,70699.9,70750.0,70699.9,70734.5,80.772,1866,70693.62,70670.96,False,70707.92,32.495492
1711395420,70734.5,70734.5,70656.3,70673.9,96.680,1870,70692.18,70689.84,False,70693.50,26.885963


In [88]:
SEQENCE_LENGTH = 32

x, y = [], []
for index in tqdm(range(SEQENCE_LENGTH, len(df))):
    fdf = df[features].iloc[index - SEQENCE_LENGTH: index]
    ndf = (fdf - fdf.min()) / (fdf.max() - fdf.min())
    # ndf = (ndf - ndf.mean()) / ndf.std()
    label = df.label.iloc[index]

    x.append(ndf.to_numpy())
    y.append(label)

x = np.array(x).reshape(-1, SEQENCE_LENGTH * len(features))
y = np.array(y)

print(x.shape, y.shape)

  0%|          | 0/9959 [00:00<?, ?it/s]

(9959, 64) (9959,)


In [89]:
TEST_LENGTH = LOOK_AHEAD
TRAIN_LENGTH = 10
WINDOW_LENGTH = TRAIN_LENGTH + TEST_LENGTH

print(TEST_LENGTH)

y_hat_list = []
correct, total = 0, 0
bar = tqdm(range(len(x)))
for start_index in bar:
    stop_index = start_index + WINDOW_LENGTH
    window_x, window_y = x[start_index: stop_index], y[start_index: stop_index]
    
    # split train and test set
    train_x, train_y = window_x[:TRAIN_LENGTH], window_y[:TRAIN_LENGTH]
    test_x, test_y = window_x[TRAIN_LENGTH:], window_y[TRAIN_LENGTH:]

    # evaluate model if there is enough samples
    if 0 < len(train_x) and 0 < len(test_x):
        # fit model
        model = LinearRegression(fit_intercept=False).fit(train_x, train_y)
        
        # evaluate model
        test_y_hat = model.predict(test_x)
        test_prediction = 0.5 < test_y_hat
        
        total += 1
        correct += (test_y[0] == test_prediction[0]).sum()
        accuracy= correct / total * 100
        bar.set_postfix_str("Accuracy: {:.2f}".format(accuracy))

        # print(test_y_hat.shape)
        y_hat_list.append(test_y_hat[0])

5


  0%|          | 0/9959 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [130]:
tdf = df[SEQENCE_LENGTH + TRAIN_LENGTH:].drop(columns=features)
tdf["prediction"] = 0.5 < np.array(y_hat_list)

tdf["fmax"] = tdf.close.rolling(21).max().shift(-LOOK_AHEAD + 1)
tdf["fmin"] = tdf.close.rolling(21).min().shift(-LOOK_AHEAD + 1)
tdf["ltp"] =  tdf.fmax / tdf.open - 1
tdf["stp"] =  tdf.open / tdf.fmin - 1

tdf["tr"] = tdf.high / tdf.low - 1
tdf["roc"] = tdf.close / tdf.open - 1
tdf["atr"] = tdf.tr.rolling(window=10).mean()
tdf["aroc"] = np.abs(tdf.roc).rolling(window=10).mean()


tdf["buy"] = tdf.prediction
tdf["sell"] = ~tdf.prediction
tdf["tp"] = tdf.aroc
tdf = tdf.dropna()

print((tdf.label == tdf.prediction).sum() / len(tdf))
print(tdf[tdf.prediction].ltp.mean())
print(tdf[~tdf.prediction].stp.mean())

print((tdf[tdf.buy].tp <= tdf[tdf.buy].ltp).sum() / tdf.buy.sum())
print((tdf[tdf.sell].tp <= tdf[tdf.sell].stp).sum() / tdf.sell.sum())

tdf

0.8451002115016618
0.003127893397247182
0.003191161108733281
0.9243546127676606
0.9389699918896999


,open,high,low,close,volume,trade,sma,fsma,label,prediction,...,fmin,ltp,stp,tr,roc,atr,aroc,buy,sell,tp
timestamp,,,,,,,,,,,,,,,,,,,,,
1710799620,67594.2,67609.0,67564.8,67594.1,55.132,1133,67602.54,67625.04,True,True,...,67524.8,0.001311,0.001028,0.000654,-0.000001,0.000774,0.000276,True,False,0.000276
1710799680,67594.1,67609.0,67573.0,67595.5,50.411,951,67607.82,67632.62,True,True,...,67524.8,0.001312,0.001026,0.000533,0.000021,0.000779,0.000262,True,False,0.000262
1710799740,67595.5,67636.0,67590.9,67623.1,48.141,861,67610.20,67630.44,True,True,...,67524.8,0.001292,0.001047,0.000667,0.000408,0.000808,0.000290,True,False,0.000290
1710799800,67623.1,67632.5,67616.0,67632.4,38.190,920,67625.04,67640.62,True,True,...,67524.8,0.000883,0.001456,0.000244,0.000138,0.000748,0.000284,True,False,0.000284
1710799860,67632.4,67643.8,67591.4,67605.9,59.558,1266,67632.62,67630.86,False,True,...,67524.8,0.000753,0.001593,0.000775,-0.000392,0.000666,0.000190,True,False,0.000190
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1711395060,70666.4,70678.3,70622.0,70677.4,139.282,2317,70690.74,70707.92,True,True,...,70570.4,0.001886,0.001360,0.000797,0.000156,0.001040,0.000575,True,False,0.000575
1711395120,70677.4,70746.0,70677.4,70746.0,126.451,2143,70689.76,70693.50,True,True,...,70570.4,0.001730,0.001516,0.000971,0.000971,0.001060,0.000614,True,False,0.000614
1711395180,70746.0,70746.0,70664.4,70693.6,76.203,1851,70696.50,70693.62,False,True,...,70570.4,0.000759,0.002488,0.001155,-0.000741,0.001064,0.000638,True,False,0.000638


In [141]:
ENR = 0.5
TPR = 2
SLR = 2

#######################
###### backtest #######
#######################
bars_list, profits_list = [], []

order_index, order_price, entry_price, entry_index = None, None, None, None
for index in tqdm(range(len(tdf))):
    if entry_price is None:
        # reset order
        if order_index is not None and order_index + 10 < index:
            order_index = None
            order_price = None

        # set order
        if order_price is None and tdf.buy.iloc[index]:
            order_index = index
            order_price = tdf.close.iloc[index] * (1 - tdf.aroc.iloc[index] * ENR)

            # print(tdf.close.iloc[index])
            # print(order_price)
            # print()
            
        # check order status
        if order_price is not None and tdf.low.iloc[index] <= order_price:
            entry_index = index
            entry_price = order_price
            
            stop_loss = tdf.aroc.iloc[index] * SLR
            take_profit = tdf.aroc.iloc[index] * TPR
    
            stop_loss_price = round(entry_price * (1 - stop_loss - 0.0002 * 2), 2)
            take_profit_price = round(entry_price * (1 + take_profit + 0.0002 * 2), 2)
            
            order_index = None
            order_price = None
            
    else:
        profit = None
        if tdf.low.iloc[index] <= stop_loss_price:
            profit = -stop_loss
        elif take_profit_price <= tdf.high.iloc[index]:
            profit = take_profit

        if profit is not None:
            profits_list.append(profit)
            bars_list.append(index - entry_index)
            
            entry_index = None
            entry_price = None

  0%|          | 0/9929 [00:00<?, ?it/s]

In [142]:
LEVERAGE = 20

bars = np.array(bars_list)
profits = np.array(profits_list) * LEVERAGE

days = len(tdf) * TimeFrame.MIN1 // TimeFrame.DAY1
min_bars = bars.min()
max_bars = bars.max()
mean_bars = bars.mean()

trades = len(profits)
daily_trades = len(profits) // days

net_profit = profits.sum()
compound_profit = (1 + profits).prod() - 1
daily_net_profit = net_profit / days
daily_compound_profit = (compound_profit + 1) ** (1 / days) - 1

wining_ratio = (0 < profits).sum() / len(profits) * 100

# print statements
print("{:<32}{}".format("Days", days))
print()
print("{:<32}{}".format("Min bars", min_bars))
print("{:<32}{:}".format("Max bars", max_bars))
print("{:<32}{:.2f}".format("Mean bars", mean_bars))
print()
print("{:<32}{}".format("Trades", trades))
print("{:<32}{}".format("Daily trades", daily_trades))
print()
print("{:<32}{:.2f}".format("Net profit", net_profit))
print("{:<32}{:.2f}".format("Compound profit", compound_profit))
print("{:<32}{:.2f}".format("Daily net profit", daily_net_profit))
print("{:<32}{:.2f}".format("Daily compound profit", daily_compound_profit))
print()
print("{:<32}{:.2f}%".format("Wining ratio", wining_ratio))


Days                            6

Min bars                        1
Max bars                        41
Mean bars                       5.01

Trades                          1008
Daily trades                    168

Net profit                      16.88
Compound profit                 10995365.80
Daily net profit                2.81
Daily compound profit           13.91

Wining ratio                    75.89%
